# Customizing Colorbars — with `xy.pyplot`

Code cells adapted from the [Python Data Science Handbook](https://github.com/jakevdp/PythonDataScienceHandbook) by Jake VanderPlas (MIT-licensed code; the book's prose is omitted).
The only systematic change is the import: `matplotlib.pyplot` → `xy.pyplot`, plus the same style-name/API modernizations the originals need to run on matplotlib ≥ 3.9 today.


# Customizing Colorbars

In [ ]:
import xy.pyplot as plt

plt.style.use("default")

In [ ]:
import numpy as np

In [ ]:
x = np.linspace(0, 10, 1000)
I = np.sin(x) * np.cos(x[:, np.newaxis])

plt.imshow(I)
plt.colorbar();

## Customizing Colorbars

In [ ]:
plt.imshow(I, cmap="Blues");

### Choosing the Colormap

In [ ]:
from xy.pyplot import LinearSegmentedColormap  # was matplotlib.colors


def grayscale_cmap(cmap):
    """Return a grayscale version of the given colormap"""
    cmap = plt.get_cmap(cmap)
    colors = cmap(np.arange(cmap.N))

    # Convert RGBA to perceived grayscale luminance
    # cf. http://alienryderflex.com/hsp.html
    RGB_weight = [0.299, 0.587, 0.114]
    luminance = np.sqrt(np.dot(colors[:, :3] ** 2, RGB_weight))
    colors[:, :3] = luminance[:, np.newaxis]

    return LinearSegmentedColormap.from_list(cmap.name + "_gray", colors, cmap.N)


def view_colormap(cmap):
    """Plot a colormap with its grayscale equivalent"""
    cmap = plt.get_cmap(cmap)
    colors = cmap(np.arange(cmap.N))

    cmap = grayscale_cmap(cmap)
    grayscale = cmap(np.arange(cmap.N))

    fig, ax = plt.subplots(2, figsize=(6, 2), subplot_kw=dict(xticks=[], yticks=[]))
    ax[0].imshow([colors], extent=[0, 10, 0, 1])
    ax[1].imshow([grayscale], extent=[0, 10, 0, 1])

In [ ]:
view_colormap("jet")

In [ ]:
view_colormap("viridis")

In [ ]:
view_colormap("RdBu")

### Color Limits and Extensions

In [ ]:
# make noise in 1% of the image pixels
speckles = np.random.random(I.shape) < 0.01
I[speckles] = np.random.normal(0, 3, np.count_nonzero(speckles))

plt.figure(figsize=(10, 3.5))

plt.subplot(1, 2, 1)
plt.imshow(I, cmap="RdBu")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(I, cmap="RdBu")
plt.colorbar(extend="both")
plt.clim(-1, 1)

### Discrete Colorbars

In [ ]:
plt.imshow(I, cmap=plt.get_cmap("Blues", 6))
plt.colorbar(extend="both")
plt.clim(-1, 1);

## Example: Handwritten Digits

In [ ]:
# load images of the digits 0 through 5 and visualize several of them
from sklearn.datasets import load_digits

digits = load_digits(n_class=6)

fig, ax = plt.subplots(8, 8, figsize=(6, 6))
for i, axi in enumerate(ax.flat):
    axi.imshow(digits.images[i], cmap="binary")
    axi.set(xticks=[], yticks=[])

In [ ]:
# project the digits into 2 dimensions using Isomap
from sklearn.manifold import Isomap

iso = Isomap(n_components=2, n_neighbors=15)
projection = iso.fit_transform(digits.data)

In [ ]:
# plot the results
plt.scatter(
    projection[:, 0], projection[:, 1], lw=0.1, c=digits.target, cmap=plt.get_cmap("plasma", 6)
)
plt.colorbar(ticks=range(6), label="digit value")
plt.clim(-0.5, 5.5)